# Assignment 2 — Part A: Count-Based Representations
**Course**: Introduction to Natural Language Processing  
**Topic**: Social Media Sentiment Analysis — Word Representations (Phase 2)  
**Team**: Team Alpha (`nlp-project-teamalpha`)  

This notebook covers:
- **Task A1**: Bag of Words (BoW) Document-Term Count Matrix, Shape, and Sparsity.
- **Task A2**: TF-IDF Matrix, Document-level Weights, and Global High vs Low IDF Analysis.
- **Task A3**: N-grams (Unigrams, Bigrams, Trigrams, Combinations) and Vocabulary Growth.
- **Task A4**: Hand-worked Numerical Calculation (TF-IDF & Bigram) matching `scikit-learn` output.


In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Load Cleaned Corpus from Phase 1
data_path = "../data/cleaned_corpus.csv" if os.path.exists("../data/cleaned_corpus.csv") else "data/cleaned_corpus.csv"
df = pd.read_csv(data_path)
docs = df['cleaned_text'].fillna('').astype(str).tolist()
print(f"Loaded {len(docs):,} cleaned documents successfully.")
df.head()


## Task A1: Bag-of-Words (CountVectorizer)
Constructing document-term count matrix and evaluating sparsity.


In [ ]:
bow_vec = CountVectorizer()
bow_matrix = bow_vec.fit_transform(docs)
n_docs, vocab_size = bow_matrix.shape
total_elements = n_docs * vocab_size
non_zero = bow_matrix.nnz
sparsity = (1.0 - (non_zero / total_elements)) * 100.0

print(f"BoW Matrix Shape:    {n_docs} documents x {vocab_size} terms")
print(f"Total Entries:       {total_elements:,}")
print(f"Non-Zero Entries:    {non_zero:,}")
print(f"Matrix Sparsity:     {sparsity:.4f}%")


## Task A2: TF-IDF Matrix & Word Weight Analysis
Evaluating how common words are driven to low IDF and discriminative words receive high weights.


In [ ]:
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(docs)
tfidf_vocab = tfidf_vec.get_feature_names_out()

# Inspect Document Index 1
sample_idx = 1
sample_vector = tfidf_matrix[sample_idx].toarray().flatten()
non_zero_indices = sample_vector.nonzero()[0]
doc_weights = sorted([(tfidf_vocab[i], sample_vector[i]) for i in non_zero_indices], key=lambda x: x[1], reverse=True)

print(f"Cleaned Text: "{docs[sample_idx]}"\n")
print(f"{'Word':<15} | {'TF-IDF Weight':<15} | {'Doc Frequency'}")
print("-" * 50)
for w, wt in doc_weights:
    df_cnt = np.sum(bow_matrix[:, bow_vec.vocabulary_[w]].toarray() > 0)
    print(f"{w:<15} | {wt:<15.4f} | {df_cnt} / {n_docs}")


## Task A3: N-grams & Vocabulary Growth
Analyzing feature explosion and sparsity trade-off across $n$-gram schemes.


In [ ]:
ngram_configs = [
    ("Unigrams (1,1)", (1, 1)),
    ("Bigrams (2,2)", (2, 2)),
    ("Trigrams (3,3)", (3, 3)),
    ("Unigrams + Bigrams (1,2)", (1, 2)),
    ("Unigrams + Bigrams + Trigrams (1,3)", (1, 3))
]

results = []
for label, n_range in ngram_configs:
    vec = CountVectorizer(ngram_range=n_range)
    mat = vec.fit_transform(docs)
    sp = (1.0 - (mat.nnz / (mat.shape[0] * mat.shape[1]))) * 100.0
    results.append({
        "N-Gram Configuration": label,
        "Vocabulary Size": len(vec.vocabulary_),
        "Non-Zero Count": mat.nnz,
        "Sparsity (%)": sp,
        "Avg Non-Zero / Doc": mat.nnz / n_docs
    })

pd.DataFrame(results)


## Task A4: Hand-Worked Numerical Calculation
Exact step-by-step arithmetic on toy sub-sample (3 documents) matching `scikit-learn` library output.

*(Note: See `HANDWRITTEN_NUMERICAL_GUIDE.md` for the printable CamScanner worksheet).*


In [ ]:
# Toy Sub-Sample
toy_docs = [
    "apple release new phone",     # d_0
    "google announce new phone",   # d_1
    "apple phone battery good"     # d_2
]
N = 3

# Hand calculations:
# IDF(t) = ln((1+N)/(1+DF(t))) + 1 = ln(4 / (1+DF)) + 1
# 'release': DF=1 -> IDF = ln(4/2) + 1 = ln(2) + 1 = 1.693147
# 'apple':   DF=2 -> IDF = ln(4/3) + 1 = 1.287682
# 'new':     DF=2 -> IDF = ln(4/3) + 1 = 1.287682
# 'phone':   DF=3 -> IDF = ln(4/4) + 1 = 1.000000

# Raw TF-IDF for d_0:
# raw('apple') = 1 * 1.287682 = 1.287682
# raw('new') = 1.287682
# raw('phone') = 1.000000
# raw('release') = 1.693147

# L2 Norm: sqrt(1.287682^2 + 1.287682^2 + 1.0^2 + 1.693147^2) = 2.680111
# Normalized TF-IDF('release', d_0) = 1.693147 / 2.680111 = 0.631745

# Verification against scikit-learn:
tfidf_toy = TfidfVectorizer(norm='l2', smooth_idf=True)
toy_mat = tfidf_toy.fit_transform(toy_docs)
toy_vocab = tfidf_toy.get_feature_names_out().tolist()

sk_val = toy_mat[0, toy_vocab.index('release')]
print(f"Hand-Calculated TF-IDF('release', d_0): 0.631745")
print(f"Scikit-Learn TF-IDF('release', d_0):    {sk_val:.6f}")
print(f"Absolute Match Error:                  {abs(0.631745 - sk_val):.2e}")


In [ ]:
# Bigram 'new phone' verification:
bigram_vec = CountVectorizer(ngram_range=(2,2))
bigram_mat = bigram_vec.fit_transform(toy_docs)
bg_vocab = bigram_vec.get_feature_names_out().tolist()
bg_idx = bg_vocab.index('new phone')

print("Bigram 'new phone' Counts per Document:")
print(f"d_0: {bigram_mat[0, bg_idx]} (Hand: 1)")
print(f"d_1: {bigram_mat[1, bg_idx]} (Hand: 1)")
print(f"d_2: {bigram_mat[2, bg_idx]} (Hand: 0)")
